# PathSnap — Phase 1: Dataset Exploration & Feature Engineering (Member 1)

This notebook covers Member 1's pre-lunch deliverables:
1. Raw dataset inspection across **80 `train` trips** and **10 `student_test` trips** (`evaluator_test` kept completely untouched).
2. Data-quality checks (missing values, 1 Hz timestamps, GPS gaps, road-network connectivity, and high-offset anomalies).
3. Coordinate projection to local meters, monotonic DP alignment between `.track` and `.route`, and interactive Folium maps.
4. KDTree candidate generation (`K=10`), 16 spatial/kinematic features, and leakage-free `StandardScaler` fit strictly on `train`.

In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display, Image

REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, REPO_ROOT)
from src.data.data_loader import load_trip, project_to_local_meters, align_track_to_route

print("Repository root:", REPO_ROOT)

## 1. Dataset Inventory & Data Quality Summary
Tables generated by `src/data/eda_analysis.py` across all 90 available trips.

In [ ]:
inv_df = pd.read_csv(os.path.join(REPO_ROOT, "outputs", "eda", "tables", "dataset_inventory.csv"))
dq_df = pd.read_csv(os.path.join(REPO_ROOT, "outputs", "eda", "tables", "data_quality_report.csv"))
trips_df = pd.read_csv(os.path.join(REPO_ROOT, "outputs", "eda", "tables", "trip_level_stats.csv"))

print("--- Dataset Inventory ---")
display(inv_df)
print("--- Data Quality Report ---")
display(dq_df)

## 2. Inspecting Trip File Structure & Anomaly Trips
Notice that `.track` rows (1 Hz GPS points) do not correspond 1-to-1 with `.route` rows (traversed `arc_id` sequence). Also notice trips `00000033` and `00000054` in `train` exhibit >120 m median GPS-to-route offset.

In [ ]:
sample = load_trip(os.path.join(REPO_ROOT, "data", "raw"), "student_test", "00000000")
print(f"Sample student_test/00000000 -> track: {sample['track'].shape}, nodes: {sample['nodes'].shape}, arcs: {sample['arcs'].shape}, route: {sample['route'].shape}")

print("\nTop 5 trips by median GPS-to-route offset (m):")
display(trips_df.sort_values("median_gps_to_route_dist_m", ascending=False).head(5))

## 3. Exploratory Plots & Folium Maps

In [ ]:
plot_files = [
    "01_trip_length_and_duration.png",
    "02_sampling_intervals_and_gaps.png",
    "03_road_network_and_gps_noise.png",
    "04_sample_trips_overlay.png",
]
for fname in plot_files:
    print("Plot:", fname)
    display(Image(filename=os.path.join(REPO_ROOT, "outputs", "eda", "plots", fname)))

In [ ]:
maps_dir = os.path.join(REPO_ROOT, "outputs", "eda", "maps")
for mfile in sorted(os.listdir(maps_dir)):
    full_p = os.path.join(maps_dir, mfile)
    print(f"Folium map: {mfile} ({os.path.getsize(full_p) / 1024:.1f} KB)")

## 4. Candidate Generation, Feature Schema & Scaler Verification

In [ ]:
with open(os.path.join(REPO_ROOT, "artifacts", "feature_schema.json"), "r", encoding="utf-8") as f:
    schema = json.load(f)

print(json.dumps(schema, indent=2))

scaler = joblib.load(os.path.join(REPO_ROOT, "artifacts", "feature_scaler.joblib"))
test_df = joblib.load(os.path.join(REPO_ROOT, "data", "processed", "student_test_candidates.pkl"))
feat_cols = schema["feature_columns"]

X_test_scaled = scaler.transform(test_df[feat_cols].values)
print("student_test_candidates shape:", test_df.shape)
print("NaNs in scaled test features:", int(np.isnan(X_test_scaled).sum()))
display(test_df.head(10))

display(Image(filename=os.path.join(REPO_ROOT, "outputs", "eda", "plots", "05_feature_distributions_and_correlations.png")))